# LTSR Colab Phase 1 — synthetic data

Phase 1–3の再現・診断用。GPU本実験の設定選択には使わない。

実行順は、Python 3.10確認 → Drive mount/source固定 → Phaseセルである。
Python環境導入でruntimeが再起動した場合は、再接続して最初のセルからやり直す。

In [ ]:
# 必ず最初に実行する。Colab UI kernelとは別に研究コード用Python 3.10を用意する。
import hashlib
import subprocess
import sys
import urllib.request
from pathlib import Path

PY310_ROOT = Path("/content/ltsr-py310")
PY310 = PY310_ROOT / "bin" / "python"
INSTALLER = Path("/content/Miniconda3-py310_23.11.0-2-Linux-x86_64.sh")
INSTALLER_URL = (
    "https://repo.anaconda.com/miniconda/"
    "Miniconda3-py310_23.11.0-2-Linux-x86_64.sh"
)
INSTALLER_SHA256 = (
    "35a58b8961e1187e7311b979968662c6223e86e1451191bed2e67a72b6bd0658"
)

def worker_version():
    if not PY310.is_file():
        return None
    return subprocess.check_output(
        [
            str(PY310), "-c",
            "import sys; print('.'.join(map(str, sys.version_info[:3])))",
        ],
        text=True,
    ).strip()

version = worker_version()
print("Colab controller:", sys.version)
print("LTSR worker before setup:", version)
if version is None or not version.startswith("3.10."):
    if not INSTALLER.is_file():
        urllib.request.urlretrieve(INSTALLER_URL, INSTALLER)
    digest = hashlib.sha256(INSTALLER.read_bytes()).hexdigest()
    if digest != INSTALLER_SHA256:
        raise RuntimeError(
            f"Miniconda installer checksum mismatch: {digest}"
        )
    subprocess.run(
        [
            "bash", str(INSTALLER), "-b", "-u",
            "-p", str(PY310_ROOT),
        ],
        check=True,
    )
    version = worker_version()
if version is None or not version.startswith("3.10."):
    raise RuntimeError(f"Python 3.10 worker setup failed: {version}")
print("LTSR worker Python 3.10: OK —", version)

In [ ]:
# Google認証とDrive mountはユーザー自身が行う。
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LTSR_colab")
REPO_ROOT = Path("/content/LTSR")
BRANCH = "20260726/gpu-scale-prep-colab"
REPO_URL = (
    "https://github.com/blabo25226/"
    "Layer-selective_Transformer-based_Symbolic_Regression.git"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
lock_path = DRIVE_ROOT / "source_lock.json"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if lock_path.is_file():
    locked_commit = json.loads(lock_path.read_text(encoding="utf-8"))["commit"]
    subprocess.run(["git", "checkout", "--detach", locked_commit], cwd=REPO_ROOT, check=True)
else:
    locked_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)

sys.path.insert(0, str(REPO_ROOT / "src"))
from colab_runtime import assert_locked_source, require_python_310

require_python_310(PY310)
print("locked commit:", assert_locked_source(REPO_ROOT, DRIVE_ROOT))
print("Drive root:", DRIVE_ROOT)

In [ ]:
# /content is ephemeral, so every fresh Colab VM must restore worker packages.
import subprocess

dependency_probe = subprocess.run(
    [
        str(PY310), "-c",
        "import numpy, torch, pytorch_lightning, nesymres, pytest, pysr",
    ],
    cwd=REPO_ROOT,
)
if dependency_probe.returncode != 0:
    commands = [
        [str(PY310), "-m", "pip", "install", "--upgrade", "pip"],
        [
            str(PY310), "-m", "pip", "install", "torch==2.5.1",
            "--index-url", "https://download.pytorch.org/whl/cu124",
        ],
        [str(PY310), "-m", "pip", "install", "-r", "requirements/gpu.txt"],
        [str(PY310), "-m", "pip", "install", "-e", "NSRS/src"],
        [str(PY310), "-m", "pip", "install", "pytest", "pysr"],
    ]
    for command in commands:
        print("+", " ".join(command), flush=True)
        subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Python 3.10 worker dependencies: already installed")

In [ ]:
from colab_runtime import restore_artifacts, restore_static_assets, run_command, sync_artifacts

DIAGNOSTIC_RUN_ID = "colab_diagnostic_20260726_01"
restore_static_assets(REPO_ROOT, DRIVE_ROOT)
restore_artifacts(REPO_ROOT, DRIVE_ROOT, DIAGNOSTIC_RUN_ID)
diagnostic_dir = REPO_ROOT / "results" / "runs" / DIAGNOSTIC_RUN_ID
phase1_data = diagnostic_dir / "input_data" / "phase1_v1"
env = {
    "LTSR_RUN_DIR": str(diagnostic_dir),
    "LTSR_PHASE1_DATA": str(phase1_data),
    "LTSR_WEIGHTS": str(REPO_ROOT / "NSRS" / "weights" / "100M.ckpt"),
    "LTSR_CONFIG": str(REPO_ROOT / "NSRS" / "jupyter" / "100M" / "config.yaml"),
    "LTSR_EQ_SETTING": str(REPO_ROOT / "NSRS" / "jupyter" / "100M" / "eq_setting.json"),
    "LTSR_PHASE_TAG": "colab",
}
command = ['python', 'scripts/issue6_generate_synthetic.py']
if command[0] == "python":
    command[0] = str(PY310)
run_command(REPO_ROOT, command, extra_env=env)
print(sync_artifacts(REPO_ROOT, DRIVE_ROOT, DIAGNOSTIC_RUN_ID))